# NB01 — Data Ingestion & Daily Smart Update

**Single notebook** that handles BOTH initial data pull AND daily incremental updates.

### Logic
1. Check if `master_daily.parquet` exists and its **last date** is within 24 hours of today
2. **If fresh** → skip API calls, load from local cache
3. **If stale or missing** → fetch from APIs (Open-Meteo, NASA FIRMS, GEE), merge, save

### Data Sources
- **Open-Meteo Archive** — ERA5 + ERA5-Land hourly weather (2012–present)
- **NASA FIRMS** — MODIS C6.1 + VIIRS C2 fire detections (local CSVs + API)
- **Google Earth Engine** — MODIS MCD64A1 burned area, Sentinel-2 NDVI/NDBI, SRTM elevation
- **OSMnx** — road network density
- **Legacy CSVs** — fallback for all GEE data

### Output
- `data/processed/master_daily.parquet` — the unified daily dataset

In [1]:
# ─── Cell 1: Install & Import ─────────────────────────────────────────────
import subprocess, sys
_pkgs = [
    "pandas", "numpy", "pyarrow", "tqdm",
    "openmeteo-requests", "requests-cache", "retry-requests",
    "requests", "folium", "matplotlib", "seaborn",
]
for _p in _pkgs:
    try:
        __import__(_p.replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

import os, sys, time, warnings, json, gc
from pathlib import Path
from datetime import datetime, timedelta, timezone
from io import StringIO

import numpy as np
import pandas as pd
import requests
import openmeteo_requests
import requests_cache
from retry_requests import retry
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


In [2]:
# ─── Cell 2: Project root & directory setup ───────────────────────────────
def _detect_project_root() -> Path:
    if os.environ.get("MANHEIM_ROOT"):
        return Path(os.environ["MANHEIM_ROOT"]).expanduser().resolve()
    if "google.colab" in sys.modules:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/MANHEIM_Data")
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return here.parent if here.name == "notebooks" else here

ROOT       = _detect_project_root()
RAW        = ROOT / "data" / "raw"
LEGACY     = RAW  / "legacy"
FIRMS_DIR  = RAW  / "firms"
PROCESSED  = ROOT / "data" / "processed"
REFERENCE  = ROOT / "data" / "reference"
OUTPUTS    = ROOT / "outputs"
MODELS     = ROOT / "models"
for p in (RAW, LEGACY, FIRMS_DIR, PROCESSED, REFERENCE, OUTPUTS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

MASTER_PATH = PROCESSED / "master_daily.parquet"
print(f"Project root : {ROOT}")
print(f"Master path  : {MASTER_PATH}")

Project root : /home/manheim666/Desktop/WildFire-Prediction
Master path  : /home/manheim666/Desktop/WildFire-Prediction/data/processed/master_daily.parquet


In [3]:
# ─── Cell 3: Constants — cities, API config, weather variables ─────────────
CITIES = {
    "Shabran":     (41.2058, 48.9878), "Baku":        (40.4093, 49.8671),
    "Ganja":       (40.6828, 46.3606), "Mingachevir":  (40.7639, 47.0595),
    "Shirvan":     (39.9317, 48.9299), "Lankaran":     (38.7523, 48.8475),
    "Shaki":       (41.1975, 47.1694), "Nakhchivan":   (39.2089, 45.4122),
    "Yevlakh":     (40.6183, 47.1500), "Quba":         (41.3611, 48.5261),
    "Khachmaz":    (41.4635, 48.8060), "Gabala":       (40.9982, 47.8468),
    "Shamakhi":    (40.6303, 48.6414), "Jalilabad":    (39.2089, 48.2986),
    "Barda":       (40.3744, 47.1266), "Zaqatala":     (41.6296, 46.6433),
}

HISTORY_START  = "2012-01-20"
FIRE_BUFFER_KM = 20
FRESHNESS_H    = 24          # hours — if master is younger than this, skip fetch

# Open-Meteo variable sets
ERA5_VARS = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure",
    "shortwave_radiation",
]
ERA5_LAND_VARS = ["soil_temperature_0_to_7cm", "soil_moisture_0_to_7cm"]
ALL_HOURLY_VARS = ERA5_VARS + ERA5_LAND_VARS

NICE_NAMES = {
    "temperature_2m": "Temperature_C", "relative_humidity_2m": "Humidity_percent",
    "precipitation": "Rain_mm", "wind_speed_10m": "Wind_Speed_kmh",
    "wind_direction_10m": "Wind_Dir_deg", "surface_pressure": "Pressure_hPa",
    "shortwave_radiation": "Solar_Radiation_Wm2",
    "soil_temperature_0_to_7cm": "Soil_Temp_C",
    "soil_moisture_0_to_7cm": "Soil_Moisture",
}

# NASA FIRMS
FIRMS_API_KEY = os.environ.get("FIRMS_MAP_KEY", "")
AZ_BBOX       = "44.77,38.38,50.39,41.91"   # west,south,east,north

today_str = datetime.now().strftime("%Y-%m-%d")
print(f"Today: {today_str}  |  Cities: {len(CITIES)}")

Today: 2026-05-05  |  Cities: 16


In [4]:
# ─── Cell 4: Master dataset check (informational only) ───────────────────
# Each data source checks its OWN freshness independently.
# This cell just loads master_daily (if it exists) for incremental-save logic later.

master_df = None

if MASTER_PATH.exists():
    master_df = pd.read_parquet(MASTER_PATH)
    master_mtime = datetime.fromtimestamp(MASTER_PATH.stat().st_mtime)
    age_hours = (datetime.now() - master_mtime).total_seconds() / 3600
    latest_date = pd.to_datetime(master_df["Date"]).max()
    days_behind = (pd.Timestamp.now() - latest_date).days

    print("master_daily.parquet found")
    print(f"  File age       : {age_hours:.1f} hours")
    print(f"  Latest date    : {latest_date.date()}")
    print(f"  Days behind    : {days_behind}")
    print(f"  Shape          : {master_df.shape}")
else:
    print("No master_daily.parquet — will build from scratch")

master_daily.parquet found
  File age       : 5.4 hours
  Latest date    : 2026-05-05
  Days behind    : 0
  Shape          : (83536, 44)


In [5]:
# ─── Cell 5: Open-Meteo weather ──────────────────────────────────────────
# Logic:
#   1. Check per-city cache freshness (modified in last 24h)
#   2. If fresh → check completeness (min date ≤ 2012-01-20, max date ≥ today)
#   3. If BOTH satisfied → load from cache
#   4. If NOT → full re-fetch from API for that city (2012-01-20 → today)
#   5. Date-alignment pass: ensure ALL cities end on the same date (today)
#
# Rate limits (free tier): 600/min, 5000/hr, 10000/day
# Strategy: 10s between cities → 16 cities in ~3min (well under 600/min)
#           If rate-limited → exponential wait 90s/180s/270s

import time as _time

cache_session = requests_cache.CachedSession(
    str(RAW / ".cache"), expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo     = openmeteo_requests.Client(session=retry_session)

TODAY = pd.Timestamp.today().normalize()
TARGET_START = pd.Timestamp("2012-01-20")
ALL_VARS = ERA5_VARS + ERA5_LAND_VARS

DELAY_BETWEEN_CITIES = 10   # seconds — keeps us under 600/min
RATE_LIMIT_WAIT      = 90   # base wait on rate limit (×attempt)


def _cache_is_valid(name):
    """Check if a city's cache is fresh AND complete.
    Returns (is_valid, reason_string).
    """
    cache_path = RAW / f"weather_hourly__{name}.parquet"
    if not cache_path.exists():
        return False, "no cache file"

    # 1) Freshness: modified within 24h
    age_hours = (_time.time() - cache_path.stat().st_mtime) / 3600
    if age_hours > 24:
        return False, f"cache is {age_hours:.0f}h old (>24h)"

    # 2) Completeness: min date ≤ 2012-01-20 AND max date ≥ today
    try:
        df = pd.read_parquet(cache_path, columns=["Timestamp"])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"]).dt.tz_localize(None)
        min_date = df["Timestamp"].min().normalize()
        max_date = df["Timestamp"].max().normalize()

        if min_date > TARGET_START + pd.Timedelta(days=1):
            return False, f"starts at {min_date.date()} (need ≤{TARGET_START.date()})"
        if max_date < TODAY - pd.Timedelta(days=1):
            return False, f"ends at {max_date.date()} (need ≥{(TODAY - pd.Timedelta(days=1)).date()})"

        return True, f"{len(df):,} rows, {min_date.date()}→{max_date.date()}"
    except Exception as e:
        return False, f"read error: {e}"


def _get_cache_max_date(name):
    """Read the max date from a city's cache file. Returns None on failure."""
    cache_path = RAW / f"weather_hourly__{name}.parquet"
    if not cache_path.exists():
        return None
    try:
        df = pd.read_parquet(cache_path, columns=["Timestamp"])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"]).dt.tz_localize(None)
        return df["Timestamp"].max().normalize()
    except Exception:
        return None


def fetch_weather_one_city(name, lat, lon, start, end, max_retries=3):
    """Fetch hourly weather for one city. Full re-fetch (start→end).
    Retries with increasing wait on rate limit.
    """
    cache_path = RAW / f"weather_hourly__{name}.parquet"

    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "hourly": ALL_VARS, "timezone": "auto",
    }

    for attempt in range(max_retries):
        try:
            resp = openmeteo.weather_api(
                "https://archive-api.open-meteo.com/v1/archive",
                params=params)[0]
            hourly = resp.Hourly()
            ts = pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()), inclusive="left"
            ).tz_localize(None)
            data = {"Timestamp": ts}
            for i, v in enumerate(ALL_VARS):
                data[NICE_NAMES.get(v, v)] = hourly.Variables(i).ValuesAsNumpy()
            df = pd.DataFrame(data)
            df["City"] = name
            df = df.sort_values("Timestamp").reset_index(drop=True)
            df.to_parquet(cache_path, index=False)
            print(f"  ✓ {name}: {len(df):,} rows  "
                  f"{df['Timestamp'].min().date()}→{df['Timestamp'].max().date()}")
            return df

        except Exception as e:
            err_msg = str(e).lower()
            if "limit" in err_msg or "rate" in err_msg or "429" in err_msg:
                wait = RATE_LIMIT_WAIT * (attempt + 1)
                print(f"  ⏳ Rate-limited {name} — waiting {wait}s "
                      f"(attempt {attempt+1}/{max_retries})")
                _time.sleep(wait)
            else:
                print(f"  ✗ {name} error: {e}")
                break

    # Exhausted retries → fall back to existing cache
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["Timestamp"] = pd.to_datetime(df["Timestamp"]).dt.tz_localize(None)
        print(f"  ⚠ {name}: API failed, using old cache ({len(df):,} rows)")
        return df
    print(f"  ✗ {name}: no data at all")
    return pd.DataFrame()


def _load_all_weather_caches():
    """Load and concat all per-city caches."""
    frames = []
    for name in CITIES:
        cache_path = RAW / f"weather_hourly__{name}.parquet"
        if cache_path.exists():
            df = pd.read_parquet(cache_path)
            df["Timestamp"] = pd.to_datetime(df["Timestamp"]).dt.tz_localize(None)
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ══════════════════════════════════════════════════════════════════════════
# MAIN: Check each city → load from cache if valid, else re-fetch
# ══════════════════════════════════════════════════════════════════════════

cities_to_fetch = []
cities_from_cache = []

print("Checking per-city caches...")
for name in CITIES:
    valid, reason = _cache_is_valid(name)
    if valid:
        cities_from_cache.append(name)
        print(f"  ✅ {name:15s} → cache OK ({reason})")
    else:
        cities_to_fetch.append(name)
        print(f"  ❌ {name:15s} → need fetch ({reason})")

print(f"\nFrom cache: {len(cities_from_cache)} | Need fetch: {len(cities_to_fetch)}")

if cities_to_fetch:
    print(f"\nFetching {len(cities_to_fetch)} cities: "
          f"{HISTORY_START} → {today_str}  ({DELAY_BETWEEN_CITIES}s delay)")
    for i, name in enumerate(cities_to_fetch):
        lat, lon = CITIES[name]
        fetch_weather_one_city(name, lat, lon, HISTORY_START, today_str)
        if i < len(cities_to_fetch) - 1:
            _time.sleep(DELAY_BETWEEN_CITIES)

    # Verify
    still_bad = []
    for name in cities_to_fetch:
        valid, reason = _cache_is_valid(name)
        if not valid:
            still_bad.append((name, reason))
    if still_bad:
        print(f"\n⚠ {len(still_bad)} cities still incomplete:")
        for name, reason in still_bad:
            print(f"    {name}: {reason}")
    else:
        print(f"\n✅ All {len(cities_to_fetch)} cities fetched successfully!")
else:
    print("\n✅ All 16 cities fresh & complete — loading from cache")

# ══════════════════════════════════════════════════════════════════════════
# DATE-ALIGNMENT PASS: ensure every city ends on the same date (today)
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("Date-alignment check: all cities must end on {today_str}...")

city_max_dates = {}
for name in CITIES:
    city_max_dates[name] = _get_cache_max_date(name)

behind_cities = []
for name, max_d in city_max_dates.items():
    if max_d is None or max_d < TODAY:
        behind_cities.append(name)
        end_str = str(max_d.date()) if max_d is not None else "N/A"
        print(f"  ⏫ {name:15s} → ends {end_str}, need {TODAY.date()}")

if behind_cities:
    print(f"\n  {len(behind_cities)} cities behind today — re-fetching...")
    for i, name in enumerate(behind_cities):
        lat, lon = CITIES[name]
        fetch_weather_one_city(name, lat, lon, HISTORY_START, today_str)
        if i < len(behind_cities) - 1:
            _time.sleep(DELAY_BETWEEN_CITIES)

    # Final verification
    final_dates = {}
    for name in CITIES:
        final_dates[name] = _get_cache_max_date(name)

    unique_ends = set(d.date() for d in final_dates.values() if d is not None)
    still_behind = [n for n, d in final_dates.items() if d is None or d < TODAY]

    if still_behind:
        print(f"\n  ⚠ {len(still_behind)} cities could not reach today:")
        for n in still_behind:
            d = final_dates[n]
            print(f"    {n}: {d.date() if d else 'N/A'}")
    else:
        print(f"\n  ✅ All {len(CITIES)} cities aligned — end dates: {unique_ends}")
else:
    unique_ends = set(d.date() for d in city_max_dates.values() if d is not None)
    print(f"  ✅ All {len(CITIES)} cities already end on today — {unique_ends}")

# Load final combined dataset
weather_hourly = _load_all_weather_caches()

print(f"\n{'='*60}")
print(f"Weather hourly: {weather_hourly.shape}")
print(f"  Cities : {weather_hourly['City'].nunique()}")
print(f"  Time   : {weather_hourly['Timestamp'].min()} → {weather_hourly['Timestamp'].max()}")

per_city = weather_hourly.groupby("City")["Timestamp"].agg(["count", "min", "max"])
for city, row in per_city.iterrows():
    flag = "✓" if row["count"] > 100000 else "⚠"
    print(f"  {flag} {city:15s} {row['count']:>8,}  "
          f"{row['min'].date()}→{row['max'].date()}")

target = len(CITIES) * len(pd.date_range(TARGET_START, TODAY, freq="h"))
coverage = len(weather_hourly) / target * 100
print(f"\n  Coverage: {len(weather_hourly):,} / {target:,} ({coverage:.1f}%)")

weather_hourly.to_parquet(RAW / "weather_hourly.parquet", index=False)
print(f"  Saved weather_hourly.parquet")

Checking per-city caches...
  ✅ Shabran         → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Baku            → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Ganja           → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Mingachevir     → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Shirvan         → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Lankaran        → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Shaki           → cache OK (125,280 rows, 2012-01-19→2026-05-05)


  ✅ Nakhchivan      → cache OK (125,280 rows, 2012-01-19→2026-05-05)


  ✅ Yevlakh         → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Quba            → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Khachmaz        → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Gabala          → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Shamakhi        → cache OK (125,280 rows, 2012-01-19→2026-05-05)


  ✅ Jalilabad       → cache OK (125,280 rows, 2012-01-19→2026-05-05)
  ✅ Barda           → cache OK (125,280 rows, 2012-01-19→2026-05-05)


  ✅ Zaqatala        → cache OK (125,280 rows, 2012-01-19→2026-05-05)

From cache: 16 | Need fetch: 0

✅ All 16 cities fresh & complete — loading from cache

────────────────────────────────────────────────────────────
Date-alignment check: all cities must end on {today_str}...


  ✅ All 16 cities already end on today — {datetime.date(2026, 5, 5)}



Weather hourly: (2004480, 11)
  Cities : 16
  Time   : 2012-01-19 20:00:00 → 2026-05-05 19:00:00
  ✓ Baku             125,280  2012-01-19→2026-05-05
  ✓ Barda            125,280  2012-01-19→2026-05-05
  ✓ Gabala           125,280  2012-01-19→2026-05-05
  ✓ Ganja            125,280  2012-01-19→2026-05-05
  ✓ Jalilabad        125,280  2012-01-19→2026-05-05
  ✓ Khachmaz         125,280  2012-01-19→2026-05-05
  ✓ Lankaran         125,280  2012-01-19→2026-05-05
  ✓ Mingachevir      125,280  2012-01-19→2026-05-05
  ✓ Nakhchivan       125,280  2012-01-19→2026-05-05
  ✓ Quba             125,280  2012-01-19→2026-05-05
  ✓ Shabran          125,280  2012-01-19→2026-05-05
  ✓ Shaki            125,280  2012-01-19→2026-05-05
  ✓ Shamakhi         125,280  2012-01-19→2026-05-05
  ✓ Shirvan          125,280  2012-01-19→2026-05-05
  ✓ Yevlakh          125,280  2012-01-19→2026-05-05
  ✓ Zaqatala         125,280  2012-01-19→2026-05-05

  Coverage: 2,004,480 / 2,004,112 (100.0%)


  Saved weather_hourly.parquet


In [6]:
# ─── Cell 6: NASA FIRMS fire data — CSV only (no API) ────────────────────
# Always loads from local CSV archives in data/raw/firms/.
# Normalised result is cached as fires_daily.parquet for speed.

fires_cache = PROCESSED / "fires_daily.parquet"


def load_firms_local():
    """Read all FIRMS CSV archives from data/raw/firms/."""
    frames = []
    for csv_file in FIRMS_DIR.rglob("*.csv"):
        if csv_file.name.lower().startswith("readme"):
            continue
        try:
            df = pd.read_csv(csv_file, low_memory=False, on_bad_lines="skip")
            frames.append(df)
        except Exception as e:
            print(f"  ⚠ Could not read {csv_file.name}: {e}")
    if not frames:
        return pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    col_map = {}
    for c in raw.columns:
        cl = c.lower().strip()
        if cl == "acq_date":                                    col_map[c] = "acq_date"
        elif cl == "latitude":                                  col_map[c] = "latitude"
        elif cl == "longitude":                                 col_map[c] = "longitude"
        elif cl in ("brightness", "bright_ti4", "bright_t31"):  col_map[c] = "brightness"
        elif cl == "frp":                                       col_map[c] = "frp"
        elif cl == "confidence":                                col_map[c] = "confidence"
    return raw.rename(columns=col_map)


def bbox_around(lat, lon, buffer_km):
    dlat = buffer_km / 111.32
    dlon = buffer_km / (111.32 * np.cos(np.radians(lat)))
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat


def normalise_fires(fires_raw):
    """Convert raw FIRMS rows to daily binary labels per city."""
    if fires_raw.empty or "acq_date" not in fires_raw.columns:
        return pd.DataFrame(columns=[
            "City", "Date", "fire_count",
            "mean_brightness", "max_frp", "Fire_Occurred"
        ])

    df = fires_raw.copy()
    df["acq_date"]  = pd.to_datetime(df["acq_date"], errors="coerce")
    df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["acq_date", "latitude", "longitude"])

    if df.empty:
        return pd.DataFrame(columns=[
            "City", "Date", "fire_count",
            "mean_brightness", "max_frp", "Fire_Occurred"
        ])

    if "brightness" not in df.columns:
        if "bright_ti4" in df.columns:
            df["brightness"] = pd.to_numeric(df["bright_ti4"], errors="coerce")
        elif "bright_t31" in df.columns:
            df["brightness"] = pd.to_numeric(df["bright_t31"], errors="coerce")
        else:
            df["brightness"] = np.nan
    else:
        df["brightness"] = pd.to_numeric(df["brightness"], errors="coerce")

    if "frp" not in df.columns:
        df["frp"] = np.nan
    else:
        df["frp"] = pd.to_numeric(df["frp"], errors="coerce")

    records = []
    for city, (clat, clon) in CITIES.items():
        w, s, e, n = bbox_around(clat, clon, FIRE_BUFFER_KM)
        mask = (
            (df["latitude"] >= s) & (df["latitude"] <= n) &
            (df["longitude"] >= w) & (df["longitude"] <= e)
        )
        city_fires = df[mask].copy()
        if city_fires.empty:
            continue
        city_fires["City"] = city
        city_fires["Date"] = city_fires["acq_date"].dt.normalize()
        records.append(city_fires)

    if not records:
        return pd.DataFrame(columns=[
            "City", "Date", "fire_count",
            "mean_brightness", "max_frp", "Fire_Occurred"
        ])

    all_city = pd.concat(records, ignore_index=True)
    daily = (
        all_city.groupby(["City", "Date"])
        .agg(
            fire_count=("latitude", "count"),
            mean_brightness=("brightness", "mean"),
            max_frp=("frp", "max")
        )
        .reset_index()
    )
    daily["Fire_Occurred"] = 1
    return daily


# ── Execute: load from parquet cache or build from CSVs ───────────────────
if fires_cache.exists():
    fires_daily = pd.read_parquet(fires_cache)
    fires_daily["Date"] = pd.to_datetime(fires_daily["Date"])
    print(f"🔥 Fire labels loaded from cache: {len(fires_daily)} events")
else:
    print("🔥 Building fire labels from local FIRMS CSVs...")
    firms_raw = load_firms_local()
    print(f"  Local FIRMS rows: {len(firms_raw)}")
    fires_daily = normalise_fires(firms_raw)
    if not fires_daily.empty:
        fires_daily.to_parquet(fires_cache, index=False)
        print(f"  ✅ Fire labels saved: {len(fires_daily)} city-day events")
    else:
        print("  ⚠ No fire detections found in CSVs")

🔥 Fire labels loaded from cache: 6745 events


In [7]:
import ee

ee.Authenticate()
ee.Initialize()

print("GEE works!")


GEE works!


In [8]:
# ─── Cell 7: GEE data — daily refresh, fallback to legacy CSVs ────────────
# Fresh = last date in CSV is today OR file was modified today.
# If stale → fetch from GEE API. Final fallback → old CSV as-is.

import time
import pandas as pd
import numpy as np
from tqdm import tqdm

TODAY_TS = pd.Timestamp.today().normalize()


def csv_is_fresh(csv_path, date_col="Date"):
    """CSV is fresh if its max Date == today OR the file was modified today."""
    if not csv_path.exists():
        return False

    # Condition 1: file was modified today
    mtime = datetime.fromtimestamp(csv_path.stat().st_mtime)
    if mtime.date() == datetime.now().date():
        return True

    # Condition 2: last date in data is today
    try:
        tmp = pd.read_csv(csv_path, usecols=[date_col])
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        max_date = tmp[date_col].max()
        if pd.notna(max_date) and max_date.normalize() >= TODAY_TS:
            return True
    except Exception:
        pass

    return False


# ── GEE init ─────────────────────────────────────────────────────────────
GEE_OK = False

try:
    import ee
    ee.Initialize()
    GEE_OK = True
    print("GEE initialised ✓")
except Exception as e:
    print("GEE not available — using legacy CSVs if needed")
    print("REAL ERROR:", repr(e))


# ── 7a. MODIS MCD64A1 Burned Area ────────────────────────────────────────
burned_csv = LEGACY / "all_cities_fire_map.csv"

if csv_is_fresh(burned_csv):
    burned_df = pd.read_csv(burned_csv)
    burned_df["Date"] = pd.to_datetime(burned_df["Date"])
    print(f"Burned area loaded from fresh CSV: {len(burned_df)}")

elif GEE_OK:
    print("CSV old/missing → fetching MODIS Burned Area from GEE...")

    robot_album = ee.ImageCollection("MODIS/061/MCD64A1").filterDate(
        HISTORY_START, (TODAY_TS + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    )

    all_burned = []

    for city_name, (lat, lon) in tqdm(CITIES.items(), desc="BurnedArea"):
        region = ee.Geometry.Point([lon, lat]).buffer(20000)

        def look_at_photo(photo):
            burn_date = photo.select("BurnDate")
            burned = burn_date.gt(0).multiply(ee.Image.pixelArea())

            stats = burned.reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=region,
                scale=500,
                maxPixels=1e9
            )

            return ee.Feature(None, {
                "date": photo.date().format("YYYY-MM-dd"),
                "size": stats.get("BurnDate")
            })

        answers = robot_album.map(look_at_photo).getInfo()["features"]

        city_df = pd.DataFrame([
            {
                "City": city_name,
                "Date": i["properties"]["date"],
                "Burned_Area_hectares": (i["properties"].get("size", 0) or 0) / 10000
            }
            for i in answers
        ])

        all_burned.append(city_df)

    burned_df = pd.concat(all_burned, ignore_index=True)
    burned_df["Date"] = pd.to_datetime(burned_df["Date"])
    burned_df.to_csv(burned_csv, index=False)

    print(f"Burned area fetched from GEE and saved: {len(burned_df)} rows")

elif burned_csv.exists():
    burned_df = pd.read_csv(burned_csv)
    burned_df["Date"] = pd.to_datetime(burned_df["Date"])
    print(f"Burned area loaded from old CSV fallback: {len(burned_df)}")

else:
    burned_df = pd.DataFrame(columns=["City", "Date", "Burned_Area_hectares"])
    print("No burned area data available")


# ── 7b. Sentinel-2 NDVI / NDBI ───────────────────────────────────────────
ndvi_csv = LEGACY / "all_cities_human_indices.csv"

if csv_is_fresh(ndvi_csv):
    ndvi_df = pd.read_csv(ndvi_csv)
    ndvi_df["Date"] = pd.to_datetime(ndvi_df["Date"])
    print(f"NDVI/NDBI loaded from fresh CSV: {len(ndvi_df)}")

elif GEE_OK:
    print("CSV old/missing → fetching Sentinel-2 NDVI/NDBI from GEE...")

    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

    all_indices = []

    for city_name, (lat, lon) in tqdm(CITIES.items(), desc="NDVI/NDBI"):
        region = ee.Geometry.Point([lon, lat]).buffer(20000)

        def get_indices(img):
            ndbi = img.normalizedDifference(["B11", "B8"]).rename("NDBI")
            ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")

            stats = ndbi.addBands(ndvi).reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=100,
                maxPixels=1e9
            )

            return ee.Feature(None, {
                "date": img.date().format("YYYY-MM-dd"),
                "NDBI": stats.get("NDBI"),
                "NDVI": stats.get("NDVI")
            })

        s2_filt = (
            s2_col
            .filterBounds(region)
            .filterDate("2020-01-01", (TODAY_TS + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        )

        ans = s2_filt.map(get_indices).getInfo()["features"]

        idx_df = pd.DataFrame([
            {
                "City": city_name,
                "Date": i["properties"]["date"],
                "NDBI": i["properties"].get("NDBI"),
                "NDVI": i["properties"].get("NDVI")
            }
            for i in ans
        ])

        all_indices.append(idx_df)

    ndvi_df = pd.concat(all_indices, ignore_index=True).dropna()
    ndvi_df["Date"] = pd.to_datetime(ndvi_df["Date"])
    ndvi_df.to_csv(ndvi_csv, index=False)

    print(f"NDVI/NDBI fetched from GEE and saved: {len(ndvi_df)} rows")

elif ndvi_csv.exists():
    ndvi_df = pd.read_csv(ndvi_csv)
    ndvi_df["Date"] = pd.to_datetime(ndvi_df["Date"])
    print(f"NDVI/NDBI loaded from old CSV fallback: {len(ndvi_df)}")

else:
    ndvi_df = pd.DataFrame(columns=["City", "Date", "NDBI", "NDVI"])
    print("No NDVI/NDBI data available")


# ── 7c. EVI vegetation index ──────────────────────────────────────────────
evi_csv = LEGACY / "all_cities_vegetation_EVI.csv"

if csv_is_fresh(evi_csv):
    evi_df = pd.read_csv(evi_csv)
    evi_df["Date"] = pd.to_datetime(evi_df["Date"])
    print(f"EVI loaded from fresh CSV: {len(evi_df)}")

elif evi_csv.exists():
    evi_df = pd.read_csv(evi_csv)
    evi_df["Date"] = pd.to_datetime(evi_df["Date"])
    print(f"EVI loaded from old CSV fallback: {len(evi_df)}")
    print("Note: this cell does not fetch EVI from GEE yet.")

else:
    evi_df = pd.DataFrame(columns=["City", "Date", "EVI"])
    print("No EVI data available")


print("\nGEE / legacy daily refresh complete.")
print("Burned max date:", burned_df["Date"].max() if not burned_df.empty else None)
print("NDVI/NDBI max date:", ndvi_df["Date"].max() if not ndvi_df.empty else None)
print("EVI max date:", evi_df["Date"].max() if not evi_df.empty else None)

GEE initialised ✓
Burned area loaded from fresh CSV: 2704
NDVI/NDBI loaded from fresh CSV: 17051
EVI loaded from old CSV fallback: 2160
Note: this cell does not fetch EVI from GEE yet.

GEE / legacy daily refresh complete.
Burned max date: 2026-02-01 00:00:00
NDVI/NDBI max date: 2026-05-04 00:00:00
EVI max date: 2026-03-22 00:00:00


In [9]:
# ─── Cell 8: Static geography + roads ────────────────────────────────────
# Elevation, Slope, Trees_pct, Urban_pct, Pop_Total + Road_Length_m, Intersections
# Auto-detects missing cities and fetches from GEE + OSMnx.

static_csv   = LEGACY / "all_cities_static_geography.csv"
roads_csv    = LEGACY / "all_cities_roads.csv"
static_cache = REFERENCE / "static_geography.parquet"

# ── 1. Load existing data ───────────────────────────────────────────────
if static_csv.exists():
    geo_df = pd.read_csv(static_csv)
    print(f"Static geo CSV: {geo_df.shape}")
else:
    geo_df = pd.DataFrame(columns=["City", "Elevation", "Slope",
                                    "Trees_pct", "Urban_pct", "Pop_Total"])
    print("No static geography CSV found")

if roads_csv.exists():
    roads_df = pd.read_csv(roads_csv)
    roads_df = roads_df.groupby("City").max().reset_index()
    print(f"Roads CSV: {roads_df.shape}")
else:
    roads_df = pd.DataFrame(columns=["City", "Road_Length_m", "Intersections"])
    print("No roads CSV found")

# ── 2. Detect missing cities ────────────────────────────────────────────
existing_geo   = set(geo_df["City"].unique())
existing_roads = set(roads_df["City"].unique())
all_cities     = set(CITIES.keys())

missing_geo    = sorted(all_cities - existing_geo)
missing_roads  = sorted(all_cities - existing_roads)

if missing_geo:
    print(f"\n⚠ Missing from static geography: {missing_geo}")
else:
    print(f"\n✅ All {len(CITIES)} cities present in static geography")

if missing_roads:
    print(f"⚠ Missing from roads: {missing_roads}")
else:
    print(f"✅ All {len(CITIES)} cities present in roads")

# ── 3. Fetch missing static geography from GEE ─────────────────────────
if missing_geo and GEE_OK:
    print(f"\nFetching static geography from GEE for: {missing_geo}")

    dem        = ee.Image('USGS/SRTMGL1_003')
    slope_img  = ee.Terrain.slope(dem)
    landcover  = ee.ImageCollection("ESA/WorldCover/v200").first()
    population = ee.ImageCollection("WorldPop/GP/100m/pop").filterDate(
        HISTORY_START, (TODAY_TS + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    ).mean()

    new_geo_rows = []
    for city_name in missing_geo:
        lat, lon = CITIES[city_name]
        print(f"  → {city_name} ({lat}, {lon})...")
        region = ee.Geometry.Point([lon, lat]).buffer(20000)

        topo = dem.addBands(slope_img).reduceRegion(
            ee.Reducer.mean(), region, 90).getInfo()
        pop = population.reduceRegion(
            ee.Reducer.sum(), region, 100).getInfo()
        lc = landcover.reduceRegion(
            ee.Reducer.frequencyHistogram(), region, 30).getInfo()

        pixels = sum(lc['Map'].values()) if 'Map' in lc else 1
        row = {
            'City':      city_name,
            'Elevation': topo.get('elevation', 0),
            'Slope':     topo.get('slope', 0),
            'Trees_pct': (lc.get('Map', {}).get('10', 0) / pixels) * 100,
            'Urban_pct': (lc.get('Map', {}).get('50', 0) / pixels) * 100,
            'Pop_Total': pop.get('population', 0),
        }
        new_geo_rows.append(row)
        print(f"    ✓ elev={row['Elevation']:.0f}m  slope={row['Slope']:.2f}  "
              f"trees={row['Trees_pct']:.1f}%  urban={row['Urban_pct']:.1f}%  "
              f"pop={row['Pop_Total']:.0f}")

    new_geo_df = pd.DataFrame(new_geo_rows)
    geo_df = pd.concat([geo_df, new_geo_df], ignore_index=True)
    geo_df.to_csv(static_csv, index=False)
    print(f"  ✅ Updated {static_csv.name}: {geo_df.shape}")

elif missing_geo:
    print("  ⚠ GEE not available — adding placeholders for missing cities")
    for city_name in missing_geo:
        geo_df = pd.concat([geo_df, pd.DataFrame([{
            'City': city_name, 'Elevation': 0, 'Slope': 0,
            'Trees_pct': 0, 'Urban_pct': 0, 'Pop_Total': 0
        }])], ignore_index=True)

# ── 4. Fetch missing roads from OSMnx ──────────────────────────────────
if missing_roads:
    try:
        import osmnx as ox

        print(f"\nFetching road networks from OSMnx for: {missing_roads}")
        new_road_rows = []
        for city_name in missing_roads:
            lat, lon = CITIES[city_name]
            print(f"  → {city_name} ({lat}, {lon})...")
            try:
                G = ox.graph_from_point((lat, lon), dist=20000,
                                        network_type="drive", simplify=True)
                stats = ox.basic_stats(G)
                new_road_rows.append({
                    'City': city_name,
                    'Road_Length_m': stats['street_length_total'],
                    'Intersections': stats['intersection_count']
                })
                print(f"    ✓ roads={stats['street_length_total']:.0f}m  "
                      f"intersections={stats['intersection_count']}")
                del G
                gc.collect()
            except Exception as e:
                print(f"    ⚠ OSMnx error: {e} — using 0")
                new_road_rows.append({
                    'City': city_name, 'Road_Length_m': 0, 'Intersections': 0
                })

        new_roads_df = pd.DataFrame(new_road_rows)
        roads_df = pd.concat([roads_df, new_roads_df], ignore_index=True)
        roads_df.to_csv(roads_csv, index=False)
        print(f"  ✅ Updated {roads_csv.name}: {roads_df.shape}")

    except ImportError:
        print("  ⚠ osmnx not installed — using 0 for missing road data")
        for city_name in missing_roads:
            roads_df = pd.concat([roads_df, pd.DataFrame([{
                'City': city_name, 'Road_Length_m': 0, 'Intersections': 0
            }])], ignore_index=True)

# ── 5. Build final static_df: merge geo + roads + lat/lon ──────────────
static_df = geo_df.merge(roads_df, on="City", how="left")

# Add Latitude/Longitude from CITIES dict
static_df["Latitude"]  = static_df["City"].map(lambda c: CITIES.get(c, (0,0))[0])
static_df["Longitude"] = static_df["City"].map(lambda c: CITIES.get(c, (0,0))[1])

static_df = static_df.fillna(0)

# ── 6. Save updated cache (always rebuild to stay in sync) ──────────────
static_df.to_parquet(static_cache, index=False)
print(f"\n✅ Static geography cache saved: {static_df.shape}")
print(static_df.to_string(index=False))

Static geo CSV: (16, 6)
Roads CSV: (16, 3)

✅ All 16 cities present in static geography
✅ All 16 cities present in roads

✅ Static geography cache saved: (16, 10)
       City   Elevation     Slope  Trees_pct  Urban_pct    Pop_Total  Road_Length_m  Intersections  Latitude  Longitude
      Ganja  498.935624  4.981345   6.375932  10.206253 4.014443e+05   3.043615e+06          11281   40.6828    46.3606
       Baku   22.952906  2.288062   1.950468  34.169409 2.048268e+06   8.707163e+06          50482   40.4093    49.8671
   Zaqatala  881.790228 16.115324  68.954934   1.625946 1.109243e+05   1.518846e+06           5404   41.6296    46.6433
   Lankaran   53.491689  4.134373  27.364711   3.866356 2.240667e+05   1.828292e+06           6140   38.7523    48.8475
      Shaki  948.448814 13.009966  33.073795   3.331254 1.115553e+05   1.514716e+06           6254   41.1975    47.1694
  Jalilabad  299.928011  7.552512   8.951281   2.751703 1.338332e+05   1.345779e+06           4535   39.2089    48.29

In [10]:
# ─── Cell 9: Build BOTH master_hourly & master_daily ──────────────────────

# ── Helper: aggregate hourly → daily ─────────────────────────────────────
def hourly_to_daily(df_h):
    """Aggregate hourly weather to daily summaries per city."""
    df = df_h.copy()
    df["Date"] = df["Timestamp"].dt.normalize()

    agg = {}
    for col in df.columns:
        if col in ("Timestamp", "Date", "City"):
            continue
        if "Rain" in col:
            agg[col + "_sum"] = (col, "sum")
        elif any(k in col for k in ["Dir", "Solar"]):
            agg[col + "_mean"] = (col, "mean")
        else:
            agg[col + "_mean"] = (col, "mean")
            agg[col + "_min"]  = (col, "min")
            agg[col + "_max"]  = (col, "max")

    daily = df.groupby(["City", "Date"]).agg(**agg).reset_index()
    return daily


# ══════════════════════════════════════════════════════════════════════════
# A) MASTER HOURLY — hourly weather + daily labels broadcast to each hour
# ══════════════════════════════════════════════════════════════════════════
print("Building master_hourly...")
master_h = weather_hourly.copy()
master_h["Date"] = master_h["Timestamp"].dt.normalize()

# Merge fire labels (daily → broadcast to hourly)
if not fires_daily.empty:
    fd = fires_daily.copy()
    fd["Date"] = pd.to_datetime(fd["Date"])
    master_h = master_h.merge(fd, on=["City", "Date"], how="left")
master_h["Fire_Occurred"] = master_h.get("Fire_Occurred", 0).fillna(0).astype(int)
for col in ["fire_count", "mean_brightness", "max_frp"]:
    if col in master_h.columns:
        master_h[col] = master_h[col].fillna(0)

# Merge burned area (daily → hourly)
if not burned_df.empty:
    bd = (burned_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
          ["Burned_Area_hectares"].max().reset_index())
    master_h = master_h.merge(bd, on=["City", "Date"], how="left")
    master_h["Burned_Area_hectares"] = master_h["Burned_Area_hectares"].fillna(0)

# Merge NDVI/NDBI (daily sparse → ffill)
if not ndvi_df.empty:
    nd = (ndvi_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
          [["NDBI", "NDVI"]].mean().reset_index())
    master_h = master_h.merge(nd, on=["City", "Date"], how="left")
    master_h = master_h.sort_values(["City", "Timestamp"])
    master_h[["NDBI", "NDVI"]] = (master_h.groupby("City")[["NDBI", "NDVI"]]
                                   .ffill().bfill())

# Merge EVI (daily sparse → ffill)
if not evi_df.empty:
    ev = (evi_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
          ["EVI"].mean().reset_index())
    master_h = master_h.merge(ev, on=["City", "Date"], how="left")
    master_h = master_h.sort_values(["City", "Timestamp"])
    master_h["EVI"] = master_h.groupby("City")["EVI"].ffill().bfill()

# Merge static geography
master_h = master_h.merge(static_df, on="City", how="left")

# Calendar features (hourly resolution)
master_h["Year"]       = master_h["Timestamp"].dt.year
master_h["Month"]      = master_h["Timestamp"].dt.month
master_h["DayOfYear"]  = master_h["Timestamp"].dt.dayofyear
master_h["DayOfWeek"]  = master_h["Timestamp"].dt.dayofweek
master_h["Hour"]       = master_h["Timestamp"].dt.hour

master_h = master_h.fillna(0)
master_h = master_h.sort_values(["City", "Timestamp"]).reset_index(drop=True)
print(f"  master_hourly: {master_h.shape}")


# ══════════════════════════════════════════════════════════════════════════
# B) MASTER DAILY — aggregated hourly → daily + same merges
# ══════════════════════════════════════════════════════════════════════════
print("Aggregating hourly → daily...")
weather_daily = hourly_to_daily(weather_hourly)
print(f"  Weather daily: {weather_daily.shape}")

master = weather_daily.copy()
if not fires_daily.empty:
    fires_daily["Date"] = pd.to_datetime(fires_daily["Date"])
    master = master.merge(fires_daily, on=["City", "Date"], how="left")
master["Fire_Occurred"] = master.get("Fire_Occurred", 0).fillna(0).astype(int)
for col in ["fire_count", "mean_brightness", "max_frp"]:
    if col in master.columns:
        master[col] = master[col].fillna(0)

if not burned_df.empty:
    burned_daily = (burned_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
                    ["Burned_Area_hectares"].max().reset_index())
    master = master.merge(burned_daily, on=["City", "Date"], how="left")
    master["Burned_Area_hectares"] = master["Burned_Area_hectares"].fillna(0)

if not ndvi_df.empty:
    ndvi_daily = (ndvi_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
                  [["NDBI", "NDVI"]].mean().reset_index())
    master = master.merge(ndvi_daily, on=["City", "Date"], how="left")
    master = master.sort_values(["City", "Date"])
    master[["NDBI", "NDVI"]] = (master.groupby("City")[["NDBI", "NDVI"]]
                                 .ffill().bfill())

if not evi_df.empty:
    evi_daily = (evi_df.groupby(["City", pd.Grouper(key="Date", freq="D")])
                 ["EVI"].mean().reset_index())
    master = master.merge(evi_daily, on=["City", "Date"], how="left")
    master = master.sort_values(["City", "Date"])
    master["EVI"] = master.groupby("City")["EVI"].ffill().bfill()

master = master.merge(static_df, on="City", how="left")

master["Year"]       = master["Date"].dt.year
master["Month"]      = master["Date"].dt.month
master["DayOfYear"]  = master["Date"].dt.dayofyear
master["DayOfWeek"]  = master["Date"].dt.dayofweek

master = master.fillna(0)
master = master.sort_values(["City", "Date"]).reset_index(drop=True)

print(f"\n  master_daily:  {master.shape}")
print(f"  Columns daily: {list(master.columns)}")
print(f"  Columns hourly ({master_h.shape[1]}): {list(master_h.columns)}")
print(f"  Fire rate: {master['Fire_Occurred'].mean()*100:.2f}%")
print(f"  Date range: {master['Date'].min().date()} → {master['Date'].max().date()}")

Building master_hourly...


  master_hourly: (2004480, 34)
Aggregating hourly → daily...


  Weather daily: (83536, 23)

  master_daily:  (83536, 44)
  Columns daily: ['City', 'Date', 'Temperature_C_mean', 'Temperature_C_min', 'Temperature_C_max', 'Humidity_percent_mean', 'Humidity_percent_min', 'Humidity_percent_max', 'Rain_mm_sum', 'Wind_Speed_kmh_mean', 'Wind_Speed_kmh_min', 'Wind_Speed_kmh_max', 'Wind_Dir_deg_mean', 'Pressure_hPa_mean', 'Pressure_hPa_min', 'Pressure_hPa_max', 'Solar_Radiation_Wm2_mean', 'Soil_Temp_C_mean', 'Soil_Temp_C_min', 'Soil_Temp_C_max', 'Soil_Moisture_mean', 'Soil_Moisture_min', 'Soil_Moisture_max', 'fire_count', 'mean_brightness', 'max_frp', 'Fire_Occurred', 'Burned_Area_hectares', 'NDBI', 'NDVI', 'EVI', 'Elevation', 'Slope', 'Trees_pct', 'Urban_pct', 'Pop_Total', 'Road_Length_m', 'Intersections', 'Latitude', 'Longitude', 'Year', 'Month', 'DayOfYear', 'DayOfWeek']
  Columns hourly (34): ['Timestamp', 'Temperature_C', 'Humidity_percent', 'Rain_mm', 'Wind_Speed_kmh', 'Wind_Dir_deg', 'Pressure_hPa', 'Solar_Radiation_Wm2', 'Soil_Temp_C', 'Soil_Moistu

In [11]:
# ─── Cell 10: Atomic save — master_daily + master_hourly ─────────────────

HOURLY_PATH = PROCESSED / "master_hourly.parquet"

# ── Save master_daily ─────────────────────────────────────────────────────
if master_df is not None:
    old_max = pd.to_datetime(master_df["Date"]).max()
    new_rows = master[master["Date"] > old_max]
    if not new_rows.empty:
        combined = pd.concat([master_df, new_rows], ignore_index=True)
        combined = combined.drop_duplicates(subset=["City", "Date"])
        combined = combined.sort_values(["City", "Date"]).reset_index(drop=True)
        tmp = MASTER_PATH.with_suffix(".tmp.parquet")
        combined.to_parquet(tmp, index=False)
        tmp.rename(MASTER_PATH)
        print(f"Daily: appended {len(new_rows)} new rows → total {len(combined)}")
        master = combined
    else:
        tmp = MASTER_PATH.with_suffix(".tmp.parquet")
        master.to_parquet(tmp, index=False)
        tmp.rename(MASTER_PATH)
        print(f"Daily: re-saved (no new rows): {master.shape}")
else:
    tmp = MASTER_PATH.with_suffix(".tmp.parquet")
    master.to_parquet(tmp, index=False)
    tmp.rename(MASTER_PATH)
    print(f"Daily: saved master_daily.parquet: {master.shape}")

# ── Save master_hourly ────────────────────────────────────────────────────
tmp_h = HOURLY_PATH.with_suffix(".tmp.parquet")
master_h.to_parquet(tmp_h, index=False)
tmp_h.rename(HOURLY_PATH)
print(f"Hourly: saved master_hourly.parquet: {master_h.shape}")

# Save cities reference
cities_df = pd.DataFrame([
    {"City": c, "Latitude": lat, "Longitude": lon}
    for c, (lat, lon) in CITIES.items()])
cities_df.to_parquet(REFERENCE / "cities.parquet", index=False)

print(f"\nFinal daily:  {master.shape}  → {MASTER_PATH}")
print(f"Final hourly: {master_h.shape} → {HOURLY_PATH}")
print(f"Daily size:  {MASTER_PATH.stat().st_size / 1e6:.1f} MB")
print(f"Hourly size: {HOURLY_PATH.stat().st_size / 1e6:.1f} MB")

Daily: re-saved (no new rows): (83536, 44)


Hourly: saved master_hourly.parquet: (2004480, 34)

Final daily:  (83536, 44)  → /home/manheim666/Desktop/WildFire-Prediction/data/processed/master_daily.parquet
Final hourly: (2004480, 34) → /home/manheim666/Desktop/WildFire-Prediction/data/processed/master_hourly.parquet
Daily size:  6.9 MB
Hourly size: 42.0 MB


In [12]:
# ─── Cell 11: Validation summary ─────────────────────────────────────────
print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)

print(f"\n{'─'*30} DAILY {'─'*30}")
print(f"Cities        : {master['City'].nunique()}")
print(f"Date range    : {master['Date'].min().date()} → {master['Date'].max().date()}")
print(f"Total rows    : {len(master):,}")
print(f"Columns       : {master.shape[1]}")
print(f"Fire days     : {master['Fire_Occurred'].sum():,.0f} "
      f"({master['Fire_Occurred'].mean()*100:.2f}%)")
print(f"Missing vals  : {master.isnull().sum().sum()}")

print(f"\n{'─'*30} HOURLY {'─'*29}")
print(f"Cities        : {master_h['City'].nunique()}")
print(f"Time range    : {master_h['Timestamp'].min()} → {master_h['Timestamp'].max()}")
print(f"Total rows    : {len(master_h):,}")
print(f"Columns       : {master_h.shape[1]}")
print(f"Fire hours    : {master_h['Fire_Occurred'].sum():,.0f} "
      f"({master_h['Fire_Occurred'].mean()*100:.2f}%)")
print(f"Missing vals  : {master_h.isnull().sum().sum()}")

print("\nDaily columns:")
for i, c in enumerate(master.columns):
    print(f"  {i+1:2d}. {c}")

print("\nHourly-extra columns:", [c for c in master_h.columns if c not in master.columns])

print("\nPer-city fire rate (daily):")
fire_rates = (master.groupby("City")["Fire_Occurred"]
              .mean().sort_values(ascending=False) * 100)
for city, rate in fire_rates.items():
    print(f"  {city:15s} {rate:5.2f}%")
print("\n→ Next: open 02_EDA_FeatureEngineering.ipynb")

INGESTION SUMMARY

────────────────────────────── DAILY ──────────────────────────────
Cities        : 16
Date range    : 2012-01-19 → 2026-05-05
Total rows    : 83,536
Columns       : 44
Fire days     : 6,740 (8.07%)
Missing vals  : 0

────────────────────────────── HOURLY ─────────────────────────────
Cities        : 16
Time range    : 2012-01-19 20:00:00 → 2026-05-05 19:00:00
Total rows    : 2,004,480
Columns       : 34
Fire hours    : 161,760 (8.07%)
Missing vals  : 0

Daily columns:
   1. City
   2. Date
   3. Temperature_C_mean
   4. Temperature_C_min
   5. Temperature_C_max
   6. Humidity_percent_mean
   7. Humidity_percent_min
   8. Humidity_percent_max
   9. Rain_mm_sum
  10. Wind_Speed_kmh_mean
  11. Wind_Speed_kmh_min
  12. Wind_Speed_kmh_max
  13. Wind_Dir_deg_mean
  14. Pressure_hPa_mean
  15. Pressure_hPa_min
  16. Pressure_hPa_max
  17. Solar_Radiation_Wm2_mean
  18. Soil_Temp_C_mean
  19. Soil_Temp_C_min
  20. Soil_Temp_C_max
  21. Soil_Moisture_mean
  22. Soil_Moisture